# Capítulo 4 – Diagnóstico del Modelo Lineal
## Sistema de Bicicletas Compartidas – Dataset `hour_prepared.csv`

En este capítulo analizamos los **supuestos del modelo lineal** ajustado en el Capítulo 3 y estudiamos la calidad del ajuste mediante gráficos de diagnóstico.

Trabajaremos nuevamente con el dataset preparado `hour_prepared.csv` y ajustaremos el mismo modelo base de regresión lineal múltiple, para luego examinar:

- Residuos vs valores ajustados.
- Normalidad de los residuos.
- Homocedasticidad.
- Puntos influyentes y leverage.


## 1. Carga de librerías, datos y modelo base

Repetimos la configuración básica: cargamos el dataset preparado, definimos X e y, realizamos la partición train/test y ajustamos el modelo OLS sobre el conjunto de entrenamiento.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11


In [ ]:
# Carga del dataset preparado
data_path = "../data/hour_prepared.csv"
df_model = pd.read_csv(data_path)
df_model.head()

Definimos la variable objetivo `cnt` y la matriz de predictores `X`.


In [ ]:
target_col = "cnt"
if target_col not in df_model.columns:
    raise ValueError("La columna 'cnt' no se encuentra en 'hour_prepared.csv'.")

y = df_model[target_col].copy()
X = df_model.drop(columns=[target_col]).copy()

X.shape, y.shape

Realizamos la partición entrenamiento/prueba con la misma semilla utilizada en el Capítulo 3 (`random_state=123`).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

X_train.shape, X_test.shape

Ajustamos el modelo OLS agregando un término de intercepto.


In [ ]:
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test, has_constant='add')

ols = sm.OLS(y_train, X_train_const).fit()
ols

In [ ]:
# Predicciones y residuales en entrenamiento
y_pred_train = ols.predict(X_train_const)
residuals_train = y_train - y_pred_train
residuals_train.describe()

## 2. Residuos vs valores ajustados

Un primer gráfico de diagnóstico estándar es el de **residuos vs valores ajustados**, que permite evaluar:

- Tendencias sistemáticas (no linealidad).
- Cambios en la dispersión (heterocedasticidad).


In [ ]:
fig, ax = plt.subplots()
ax.scatter(y_pred_train, residuals_train, alpha=0.3)
ax.axhline(0, linestyle='--')
ax.set_xlabel("Valores ajustados (train)")
ax.set_ylabel("Residuos (train)")
ax.set_title("Residuos vs valores ajustados")
plt.tight_layout()
plt.show()

## 3. Normalidad de los residuos

Para evaluar la normalidad de los errores, analizamos:

- Histograma de los residuales.
- Gráfico Q–Q (cuantiles teóricos vs cuantiles observados).


In [ ]:
# Histograma de residuales
fig, ax = plt.subplots()
ax.hist(residuals_train, bins=30)
ax.set_title("Histograma de residuales (train)")
ax.set_xlabel("Residual")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico Q-Q de los residuales
fig = sm.ProbPlot(residuals_train, fit=True).qqplot(line='45')
plt.title("Gráfico Q-Q de residuales (train)")
plt.tight_layout()
plt.show()

## 4. Homocedasticidad: gráfico de escala-localización

El gráfico de **escala-localización** muestra la raíz cuadrada del valor absoluto de los residuales estandarizados frente a los valores ajustados. Permite detectar patrones de variabilidad creciente o decreciente.


In [ ]:
# Residuos estandarizados
influence = ols.get_influence()
standard_resid = influence.resid_studentized_internal

fig, ax = plt.subplots()
ax.scatter(y_pred_train, np.sqrt(np.abs(standard_resid)), alpha=0.3)
ax.set_xlabel("Valores ajustados (train)")
ax.set_ylabel("sqrt(|resid estandarizado|)")
ax.set_title("Gráfico de escala-localización")
plt.tight_layout()
plt.show()

## 5. Leverage y puntos influyentes

Utilizamos las medidas de influencia del modelo OLS para identificar posibles observaciones influyentes (alto leverage y/o alto impacto en los coeficientes).


In [ ]:
# Leverage (h_ii) y medidas de influencia
influence = ols.get_influence()
leverage = influence.hat_matrix_diag
cooks_d = influence.cooks_distance[0]

pd.DataFrame({"leverage": leverage, "cooks_d": cooks_d}).describe()

In [ ]:
# Gráfico leverage vs residuales estandarizados
fig, ax = plt.subplots()
ax.scatter(leverage, standard_resid, alpha=0.3)
ax.set_xlabel("Leverage (h_ii)")
ax.set_ylabel("Residuos estandarizados")
ax.set_title("Residuos estandarizados vs leverage")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de Cook's distance
fig, ax = plt.subplots()
ax.stem(np.arange(len(cooks_d)), cooks_d, use_line_collection=True)
ax.set_xlabel("Índice de observación")
ax.set_ylabel("Cook's distance")
ax.set_title("Cook's distance por observación")
plt.tight_layout()
plt.show()

## 6. Multicolinealidad: Factor de Inflación de la Varianza (VIF)

Para evaluar la presencia de **multicolinealidad** entre los predictores, calculamos el **Variance Inflation Factor (VIF)** para cada columna de `X_train`.

Valores de referencia típicos:

- VIF ≈ 1: sin colinealidad.
- 1–5: colinealidad moderada.
- > 5 (o > 10 según criterio): colinealidad alta.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculamos VIF sobre X_train (sin la columna de intercepto)
X_vif = X_train.copy()
vif_data = []
for i, col in enumerate(X_vif.columns):
    vif_val = variance_inflation_factor(X_vif.values, i)
    vif_data.append({"variable": col, "VIF": vif_val})

vif_df = pd.DataFrame(vif_data).sort_values("VIF", ascending=False)
vif_df.reset_index(drop=True, inplace=True)
vif_df.round(2)

## 7. Resumen del capítulo

En este capítulo realizamos un diagnóstico básico del modelo lineal:

- Graficamos **residuos vs valores ajustados** para evaluar linealidad y forma.
- Analizamos la **normalidad de los residuos** mediante histograma y gráfico Q–Q.
- Exploramos la **homocedasticidad** con un gráfico de escala-localización.
- Estudiamos **leverage** y **Cook's distance** para identificar observaciones influyentes.
- Calculamos el **VIF** para evaluar multicolinealidad entre predictores.

Estos resultados servirán de base para, en capítulos posteriores, considerar transformaciones, selección de variables y métodos más robustos o regularizados si los supuestos del modelo clásico no se cumplen adecuadamente.
